In [1]:
import os
if os.getcwd().endswith("analysis"): os.chdir("..")
print(os.getcwd())

import ipynbname
__file__ = ipynbname.path()

/mnt/mergerfs/yuzihan/MergedWorkSpace/42-MySRAgent/SRAgent


In [165]:
import json
import pandas as pd
from pathlib import Path

df_list_all = []
df_list_all2 = []
for exp_path in [
    Path('logs/bench_sr_agent/bench3_my_sr_agent_deepseek-v4-flash'),
    Path('logs/bench_sr_agent/codex_gpt55/'),
    Path('logs/bench_sr_agent/codex_gpt55_notools/'),
    Path('logs/bench_sr_agent/codex_gpt55_anonymize/'),
    Path('logs/bench_sr_agent/codex_gpt55_anonymize_notools/'),
]:
    df_list = []
    for result_json in sorted(exp_path.glob('results/*.jsonl'), key=lambda x: x.stat().st_mtime):
        result = json.loads(result_json.read_text().splitlines()[-1])
        result['result_json'] = result_json
        df_list.append(result)
    df_result = pd.DataFrame(df_list).assign(exp_name=exp_path.name)
    df_list_all2.append(df_result)
    # display(f"==={exp_path.name}===")

    df1 = df_result.groupby('dataset_identifier')['symbolic_acc'].agg(['mean', 'count'])
    df1.columns = ['symbolic_acc_mean', 'symbolic_acc_count']

    df2 = pd.concat([df_result, pd.DataFrame(df_result['id_metrics'].tolist(), index=df_result.index)], axis=1)
    df2 = df2.groupby('dataset_identifier')[['mse', 'nmse', 'r2', 'acc01', 'mape', 'num_valid_points']].mean()

    df = pd.concat([df1, df2], axis=1).assign(exp_name=exp_path.name).set_index('exp_name', append=True)
    # display(df)
    df_list_all.append(df)

df_result_all = pd.concat(df_list_all2)
df = pd.concat(df_list_all)
display(df)

symbolic_acc_mean  \
dataset_identifier exp_name                                                  
bio_pop_growth     bench3_my_sr_agent_deepseek-v4-flash           0.208333   
chem_react         bench3_my_sr_agent_deepseek-v4-flash           0.000000   
lsrtransform       bench3_my_sr_agent_deepseek-v4-flash           0.585586   
                   codex_gpt55                                    0.900901   
                   codex_gpt55_notools                            0.945455   
                   codex_gpt55_anonymize                          0.666667   
                   codex_gpt55_anonymize_notools                  0.762500   

                                                         symbolic_acc_count  \
dataset_identifier exp_name                                                   
bio_pop_growth     bench3_my_sr_agent_deepseek-v4-flash                  24   
chem_react         bench3_my_sr_agent_deepseek-v4-flash                   5   
lsrtransform       bench3_my_sr_agent_deepseek-v4-flash                 111   
                   codex_gpt55                                          111   
                   codex_gpt55_notools                                  110   
                   codex_gpt55_anonymize                                111   
                   codex_gpt55_anonymize_notools                         80   

                                                                  mse  \
dataset_identifier exp_name                                             
bio_pop_growth     bench3_my_sr_agent_deepseek-v4-flash  1.831383e-04   
chem_react         bench3_my_sr_agent_deepseek-v4-flash  8.673685e-11   
lsrtransform       bench3_my_sr_agent_deepseek-v4-flash  1.088018e+28   
                   codex_gpt55                           2.266464e+27   
                   codex_gpt55_notools                   2.287069e+27   
                   codex_gpt55_anonymize                 2.266464e+27   
                   codex_gpt55_anonymize_notools         3.144719e+27   

                                                                 nmse  \
dataset_identifier exp_name                                             
bio_pop_growth     bench3_my_sr_agent_deepseek-v4-flash  4.777716e-04   
chem_react         bench3_my_sr_agent_deepseek-v4-flash  5.884686e-07   
lsrtransform       bench3_my_sr_agent_deepseek-v4-flash  6.458157e-01   
                   codex_gpt55                           2.836098e-04   
                   codex_gpt55_notools                   4.188013e-04   
                   codex_gpt55_anonymize                 5.916655e-02   
                   codex_gpt55_anonymize_notools         1.317406e-02   

                                                               r2     acc01  \
dataset_identifier exp_name                                                   
bio_pop_growth     bench3_my_sr_agent_deepseek-v4-flash  0.999522  0.866417   
chem_react         bench3_my_sr_agent_deepseek-v4-flash  0.999999  0.985600   
lsrtransform       bench3_my_sr_agent_deepseek-v4-flash  0.354184  0.829182   
                   codex_gpt55                           0.999716  0.987536   
                   codex_gpt55_notools                   0.999581  0.992796   
                   codex_gpt55_anonymize                 0.940833  0.914512   
                   codex_gpt55_anonymize_notools         0.986826  0.972037   

                                                                mape  \
dataset_identifier exp_name                                            
bio_pop_growth     bench3_my_sr_agent_deepseek-v4-flash     0.101424   
chem_react         bench3_my_sr_agent_deepseek-v4-flash     0.011683   
lsrtransform       bench3_my_sr_agent_deepseek-v4-flash  1724.649028   
                   codex_gpt55                              9.854920   
                   codex_gpt55_notools                     32.218354   
                   codex_gpt55_anonymize                 4694.496613   
                   codex_gpt55_anonymi

In [163]:
df_result_all[~df_result_all['symbolic_acc_detail'].str.contains('LLM judgement agrees')].groupby(['exp_name'])['symbolic_acc'].agg(['mean', 'count', 'size'])

,mean,count,size
exp_name,,,
bench3_my_sr_agent_deepseek-v4-flash,0.285714,7,7
codex_gpt55,0.250000,4,4
codex_gpt55_anonymize,0.333333,3,3
codex_gpt55_anonymize_notools,0.028571,35,35
codex_gpt55_notools,0.200000,5,5


In [114]:
import json
import pandas as pd
from pathlib import Path

exp_path = Path('logs/bench_sr_agent/bench3_my_sr_agent_deepseek-v4-flash')

df_list = []
for exp_result_json in sorted(exp_path.glob('experiments/*/result.jsonl'), key=lambda x: x.stat().st_mtime):
    result = json.loads(exp_result_json.read_text())
    df_list.append(result)
df_exp_result = pd.DataFrame(df_list)

df = df_exp_result[['money_usage', 'token_usage', 'duration_seconds']].mean().to_frame('mean').T
display(df)

df = pd.DataFrame(df_exp_result['tools_usage'].to_list(), index=df_exp_result.index).fillna(0).astype(int)
df['total'] = df.sum(axis=1)
df = df.mean().to_frame('mean').T
display(df)

,money_usage,token_usage,duration_seconds
mean,0.033207,420986.3875,747.375217


,statistics_analysis,predict_property,code_executor,evaluate_formula,read_skill,call_sindy,call_pysr,polynomial_fit,submit_formula,total
mean,9.1125,5.25,15.5,23.175,0.3125,3.05,1.2125,0.825,1.9125,60.35


In [118]:
import json
import pandas as pd
from pathlib import Path

df_list_all = []

for exp_path in [
    Path('logs/bench_sr_agent/codex_gpt55/'),
    Path('logs/bench_sr_agent/codex_gpt55_notools/'),
    Path('logs/bench_sr_agent/codex_gpt55_anonymize/'),
    Path('logs/bench_sr_agent/codex_gpt55_anonymize_notools/'),
]:
    df_list = []
    for exp_result_json in sorted(exp_path.glob('experiments/*/result.json'), key=lambda x: x.stat().st_mtime):
        result = json.loads(exp_result_json.read_text())
        df_list.append(result)
    df_exp_result = pd.DataFrame(df_list)
    df_exp_result = df_exp_result.groupby('equation_id').last()

    df1 = df_exp_result[['duration_seconds', 'tool_call_count']].mean().to_frame(exp_path.name).T
    # display(df1)

    df2 = pd.DataFrame(df_exp_result['token_usage'].map(lambda x: x or {}).tolist(), index=df_exp_result.index)
    df2['total_tokens'] = df2.sum(axis=1)
    df2 = df2.mean().to_frame(exp_path.name).T
    # display(df2)

    df = pd.concat([df1, df2], axis=1)
    # display(df)
    df_list_all.append(df)

df = pd.concat(df_list_all)
df

,duration_seconds,tool_call_count,input_tokens,cached_input_tokens,output_tokens,reasoning_output_tokens,total_tokens
codex_gpt55,221.702491,13.234234,5.664142e+05,5.173120e+05,5664.572727,976.272727,1.080544e+06
codex_gpt55_notools,121.883170,1.268041,1.987588e+05,1.685034e+05,4640.099099,909.711712,3.728120e+05
codex_gpt55_anonymize,447.675805,21.927928,1.217887e+06,1.130035e+06,12630.895238,3040.419048,2.235832e+06
codex_gpt55_anonymize_notools,316.863923,1.621951,5.686063e+05,5.085617e+05,11190.701149,2959.804598,8.553577e+05


In [164]:
import json
import pandas as pd
from pathlib import Path

df_list_all = []
for exp_path in [
    Path('logs/bench_sr_agent/bench3_my_sr_agent_deepseek-v4-flash'),
    Path('logs/bench_sr_agent/codex_gpt55/'),
    Path('logs/bench_sr_agent/codex_gpt55_notools/'),
    Path('logs/bench_sr_agent/codex_gpt55_anonymize/'),
    Path('logs/bench_sr_agent/codex_gpt55_anonymize_notools/'),
]:
    df_list = []
    for result_json in sorted(exp_path.glob('results/*.jsonl'), key=lambda x: x.stat().st_mtime):
        result = json.loads(result_json.read_text().splitlines()[-1])
        result['result_json'] = result_json
        df_list.append(result)
    df_result = pd.DataFrame(df_list).assign(exp_name=exp_path.name)
    df_list_all.append(df_result)

df_result_all = pd.concat(df_list_all)

df = df_result_all.pivot_table(columns='exp_name', index='equation_id', values='symbolic_acc')
# df_result_all['symbolic_acc']
display(df[(df['codex_gpt55_notools'] > df['codex_gpt55'])])
display(df[(df['codex_gpt55_notools'] < df['codex_gpt55'])])

display(df[(df['codex_gpt55_anonymize_notools'] > df['codex_gpt55_anonymize'])])
display(df[(df['codex_gpt55_anonymize_notools'] < df['codex_gpt55_anonymize'])])

exp_name,bench3_my_sr_agent_deepseek-v4-flash,codex_gpt55,codex_gpt55_anonymize,codex_gpt55_anonymize_notools,codex_gpt55_notools
equation_id,,,,,
I.32.17_5_3,0.0,0.0,0.0,0.0,1.0
II.11.27_3_0,0.0,0.0,1.0,1.0,1.0
II.11.28_0_0,0.0,0.0,1.0,1.0,1.0
II.36.38_3_0,0.0,0.0,0.0,0.0,1.0
II.36.38_7_0,0.0,0.0,0.0,0.0,1.0
II.6.15a_4_1,0.0,0.0,1.0,0.0,1.0
III.19.51_3_0,NaN,0.0,1.0,0.0,1.0


exp_name,bench3_my_sr_agent_deepseek-v4-flash,codex_gpt55,codex_gpt55_anonymize,codex_gpt55_anonymize_notools,codex_gpt55_notools
equation_id,,,,,
II.35.18_2_1,0.0,1.0,0.0,1.0,0.0
II.6.15a_1_0,0.0,1.0,1.0,1.0,0.0
II.6.15a_2_0,0.0,1.0,0.0,1.0,0.0


exp_name,bench3_my_sr_agent_deepseek-v4-flash,codex_gpt55,codex_gpt55_anonymize,codex_gpt55_anonymize_notools,codex_gpt55_notools
equation_id,,,,,
I.11.19_4_0,1.0,1.0,0.0,1.0,1.0
I.24.6_3_1,1.0,1.0,0.0,1.0,1.0
I.27.6_2_0,1.0,1.0,0.0,1.0,1.0
I.29.16_1_0,0.0,1.0,0.0,1.0,1.0
II.11.17_5_0,0.0,1.0,0.0,1.0,1.0
II.35.18_2_1,0.0,1.0,0.0,1.0,0.0
II.35.21_2_1,1.0,1.0,0.0,1.0,1.0
II.6.15a_2_0,0.0,1.0,0.0,1.0,0.0
II.6.15a_5_0,0.0,1.0,0.0,1.0,1.0


exp_name,bench3_my_sr_agent_deepseek-v4-flash,codex_gpt55,codex_gpt55_anonymize,codex_gpt55_anonymize_notools,codex_gpt55_notools
equation_id,,,,,
I.11.19_1_0,1.0,1.0,1.0,0.0,1.0
I.11.19_5_0,NaN,1.0,1.0,0.0,1.0
I.32.17_3_0,NaN,1.0,1.0,0.0,1.0
I.40.1_4_0,NaN,1.0,1.0,0.0,1.0
I.44.4_2_0,0.0,1.0,1.0,0.0,1.0
II.11.17_2_0,NaN,1.0,1.0,0.0,1.0
II.11.27_0_0,1.0,1.0,1.0,0.0,1.0
II.11.3_0_0,NaN,1.0,1.0,0.0,1.0
II.11.3_2_0,NaN,1.0,1.0,0.0,1.0


In [145]:
df_result_all[df_result_all['exp_name'].eq('codex_gpt55_notools') & df_result_all['equation_id'].eq('I.32.17_5_3')]['result_json'].values

array([PosixPath('logs/bench_sr_agent/codex_gpt55_notools/results/lsrtransform_I.32.17_5_3.jsonl')],
      dtype=object)

In [ ]:
df_result_all[df_result_all['exp_name'].eq('codex_gpt55_notools') & df_result_all['equation_id'].eq('I.32.17_5_3')]['result_json'].values

In [135]:
Path('logs/bench_sr_agent/codex_gpt55_anonymize_notools/') / 'I.32.17_5_3'

PosixPath('logs/bench_sr_agent/codex_gpt55_anonymize_notools/I.32.17_5_3')